# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/3bud-ZC/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The playbook turns model/rule scores into a **human review queue**, not an automatic publishing system.

The ordering uses a learned decline-risk score plus the frozen transparent baseline. The action label is then chosen from feature-time reason codes that a reviewer can inspect:

- `review_for_refresh` — visible + stale or high model risk.
- `review_title_meta_and_intent` — visible, ranking within reach, but low CTR.
- `review_engagement_and_intent` — enough sessions but weak engagement/scroll.
- `review_depth_and_coverage` — visible but unusually thin.
- `monitor_only` — score is not strong enough for an edit recommendation.

The model orders the list; reason codes explain why a human should look.

In [1]:
import os, json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

REPO_URL="https://github.com/3bud-ZC/flyrank-ml-internship"
REPO_DIR="flyrank-ml-internship"

def find_repo_root():
    here=Path.cwd().resolve()
    for candidate in [here,*here.parents]:
        if (candidate/"data/raw/content_refresh_anonymized.csv").exists():
            return candidate
    return None

root=find_repo_root()
if root is None:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR],check=True)
    root=Path(REPO_DIR).resolve()
os.chdir(root)

df=pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["target"]=df["trend_direction"].str.lower().eq("down").astype(int)
features=[
    "impressions_90d","clicks_90d","sessions_90d","avg_position","ctr",
    "content_age_days","days_since_last_update","word_count",
    "engagement_rate","scroll_rate","days_with_impressions","days_with_sessions"
]

# frozen baseline
def pct_rank(s): return pd.Series(s).rank(pct=True,method="average").fillna(0).to_numpy()
visibility=pct_rank(np.log1p(df["impressions_90d"].clip(lower=0)))
freshness=pct_rank(df["days_since_last_update"].fillna(0))
pos=df["avg_position"].fillna(0).to_numpy()
position=((51-np.clip(pos,1,50))/50)*visibility*(pos>0)
depth=(1-pct_rank(df["word_count"].fillna(df["word_count"].median())))*visibility
df["baseline_score"]=np.clip(.40*visibility+.30*freshness+.25*position+.05*depth,0,1)

# full-data score used ONLY for queue ordering after validation was already completed in W5/W6
rf=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("model",RandomForestClassifier(
        n_estimators=300,max_depth=10,min_samples_leaf=25,
        class_weight="balanced_subsample",n_jobs=-1,random_state=42
    ))
])
rf.fit(df[features],df["target"])
df["model_score"]=rf.predict_proba(df[features])[:,1]
df["final_score"]=100*(.70*df["model_score"]+.30*df["baseline_score"])

def reason_codes(row):
    r=[]
    if row["days_since_last_update"]>=180 and row["impressions_90d"]>=500:r.append("stale_visible")
    if row["impressions_90d"]>=500 and 0<row["avg_position"]<=20 and row["ctr"]<0.5:r.append("low_ctr_visible")
    if pd.notna(row["word_count"]) and 0<row["word_count"]<1200 and row["impressions_90d"]>=250:r.append("thin_visible")
    if row["sessions_90d"]>=30 and (
        (row["engagement_rate"]>0 and row["engagement_rate"]<30) or
        (row["scroll_rate"]>0 and row["scroll_rate"]<30)
    ):r.append("low_engagement_visible")
    if row["model_score"]>=0.65:r.append("model_high_risk")
    return "|".join(r or ["general_review"])

def action(row):
    rs=set(row["reason_codes"].split("|"))
    if "low_ctr_visible" in rs:return "review_title_meta_and_intent"
    if "thin_visible" in rs:return "review_depth_and_coverage"
    if "low_engagement_visible" in rs:return "review_engagement_and_intent"
    if "stale_visible" in rs or "model_high_risk" in rs:return "review_for_refresh"
    return "monitor_only"

df["reason_codes"]=df.apply(reason_codes,axis=1)
df["action"]=df.apply(action,axis=1)
queue=df.sort_values(["final_score","impressions_90d"],ascending=[False,False]).copy()
queue["rank"]=np.arange(1,len(queue)+1)

print(queue[["rank","final_score","action","reason_codes","impressions_90d","avg_position","ctr"]].head(15).round(3).to_string(index=False))


 rank  final_score                       action                                           reason_codes  impressions_90d  avg_position  ctr
    1       84.239 review_title_meta_and_intent low_ctr_visible|low_engagement_visible|model_high_risk            22537          15.9 0.36
    2       84.049 review_engagement_and_intent                 low_engagement_visible|model_high_risk            25485          13.8 0.52
    3       83.365 review_engagement_and_intent                 low_engagement_visible|model_high_risk            18117          12.7 0.77
    4       83.313 review_title_meta_and_intent low_ctr_visible|low_engagement_visible|model_high_risk            21533          14.6 0.47
    5       83.289 review_engagement_and_intent                 low_engagement_visible|model_high_risk            12732          13.6 0.94
    6       83.129 review_engagement_and_intent                 low_engagement_visible|model_high_risk            12992          15.2 0.90
    7       83.042 review_t

## 2. Intended use and limits

**Intended user:** an SEO/content strategist responsible for a large portfolio with limited review capacity.

**Intended use:** start the weekly/monthly review session from the top of the queue, read the reason codes and page context, then decide whether to investigate refresh, snippet/intent alignment, engagement, depth, consolidation, or monitoring.

**Limits:**
- The current implementation is trained on the bundled 30,000-row anonymized starter slice and a same-window decline proxy.
- The score is not a probability that a refresh will work.
- It cannot distinguish every case of seasonality, consolidation, SERP-layout change, campaign effects, or instrumentation noise.
- It must not expose or attempt to infer real clients, URLs, keywords, or queries.
- A future warehouse version needs a non-overlapping past→future target and time-forward validation before production use.

### Archetype → action mapping

| Review archetype | Observable evidence | Human action |
|---|---|---|
| CTR opportunity | visible, position 1–20, low CTR | review title/meta, search intent, and SERP context |
| Stale visible page | meaningful impressions + long time since update | review accuracy, freshness, and strategic relevance |
| Engagement concern | enough sessions + weak engagement/scroll | review intent match, UX, and measurement coverage |
| Thin visible page | real visibility + unusually low depth | review topical completeness; do not add words mechanically |
| Uncertain / single-signal | weak or conflicting evidence | monitor, gather context, avoid edit automation |

### Cost / value thinking

The value of the queue comes from concentrating limited reviewer time on higher-priority candidates. A false positive costs analyst/editor time and may lead to unnecessary work; a false negative can leave a meaningful decline or opportunity unreviewed. For that reason, the playbook optimizes review order, keeps a measurable Precision@50 floor, and reserves irreversible or costly actions for human approval.

In [2]:
intended_use={
    "actor":"SEO/content strategist",
    "decision":"which pages to inspect first",
    "output":"ranked human-review queue",
    "automatic_editing":False,
    "causal_claim":False,
    "current_data_scope":"30,000-row anonymized starter slice",
    "future_requirement":"past-to-future warehouse target + time-aware validation"
}
print(json.dumps(intended_use,indent=2))


{
  "actor": "SEO/content strategist",
  "decision": "which pages to inspect first",
  "output": "ranked human-review queue",
  "automatic_editing": false,
  "causal_claim": false,
  "current_data_scope": "30,000-row anonymized starter slice",
  "future_requirement": "past-to-future warehouse target + time-aware validation"
}


## 3. Human review + the no-go list

Before acting on any recommendation, a reviewer should check:

1. Is the page still strategically relevant?
2. Is the apparent weakness persistent, or just a low-volume fluctuation?
3. Did a sibling page absorb the traffic (consolidation/cannibalization)?
4. Is seasonality or a campaign a plausible explanation?
5. For low CTR, did SERP layout/query mix change rather than the snippet?
6. Is the content already being edited by another owner?
7. Does the recommended action match user intent?

### Never automate from this score
- no automatic rewrites or publishing,
- no deletion/pruning,
- no redirects/merges,
- no title/meta changes,
- no outreach or budget allocation,
- no client-specific action without human context.

The queue prioritizes **review**, not execution.

In [3]:
no_go=[
    "auto rewrite/publish","delete/prune","redirect/merge",
    "automatic title/meta changes","automatic spend allocation",
    "client action without human context"
]
review_checks=[
    "strategic relevance","persistence vs noise","consolidation",
    "seasonality/campaigns","SERP/query-mix explanation",
    "existing editorial work","intent match"
]
print("Human checks:",len(review_checks))
print("No-go automations:",len(no_go))
assert "auto rewrite/publish" in no_go


Human checks: 7
No-go automations: 6


## 4. Monitoring / retrain triggers

The system should be reviewed or retrained when any of these occur:

- **Target/base-rate drift:** positive rate moves by more than 10 percentage points from the development reference.
- **Ranking drift:** Precision@50 falls below **0.60** or below the frozen rule for two evaluation windows.
- **Feature drift:** major input distributions (impressions, position, age/freshness) shift materially.
- **Coverage drift:** tracking/availability changes make engagement features unreliable.
- **Policy drift:** the editorial review budget or action taxonomy changes.
- **Data/schema drift:** columns, meanings, or availability flags change.

Until a future warehouse label exists, these are a proposed operating policy, not evidence of a live production monitor.

In [4]:
with open("work/outputs/w05_model_metrics.json","r",encoding="utf-8") as fh:
    w5=json.load(fh)

rf_row=next(r for r in w5["results"] if r["method"]=="random_forest")
baseline_row=next(r for r in w5["results"] if r["method"]=="fixed_rule")

monitor_policy={
    "p50_floor":0.60,
    "baseline_p50_reference":baseline_row["p50"],
    "model_p50_reference":rf_row["p50"],
    "base_rate_reference":w5["test_base_rate"],
    "base_rate_drift_absolute":0.10,
    "review_if_model_loses_to_baseline_windows":2
}
print(json.dumps(monitor_policy,indent=2))


{
  "p50_floor": 0.6,
  "baseline_p50_reference": 0.26,
  "model_p50_reference": 0.86,
  "base_rate_reference": 0.3909677419354839,
  "base_rate_drift_absolute": 0.1,
  "review_if_model_loses_to_baseline_windows": 2
}


## 5. Exports for the paper

The queue CSV stays under `work/outputs/` and remains out of git by repository policy. Small JSON receipts and SVG figures are committed so the paper's reported numbers can be traced without redistributing bulk data.

The exports intentionally contain only pseudonymized IDs and aggregate/derived fields.

In [5]:
from collections import Counter

Path("work/outputs").mkdir(parents=True,exist_ok=True)
Path("work/figures").mkdir(parents=True,exist_ok=True)

export_cols=[
    "rank","content_id","client_id","final_score","model_score","baseline_score",
    "action","reason_codes","impressions_90d","avg_position","ctr",
    "content_age_days","days_since_last_update"
]
queue[export_cols].to_csv("work/outputs/w07_action_queue.csv",index=False)

top50=queue.head(50)
metrics={
    "rows_ranked":int(len(queue)),
    "top50_proxy_precision":float(top50["target"].mean()),
    "top50_mean_score":float(top50["final_score"].mean()),
    "action_counts":{k:int(v) for k,v in queue["action"].value_counts().items()},
    "top_reason_counts":{
        k:int(v) for k,v in Counter(
            r for cell in queue["reason_codes"] for r in cell.split("|")
        ).most_common(10)
    },
    "monitor_policy":monitor_policy,
    "human_review_required":True,
    "automatic_editing":False
}
Path("work/outputs/w07_playbook_metrics.json").write_text(json.dumps(metrics,indent=2))

# self-contained SVG bar chart for paper reuse
counts=queue["action"].value_counts()
width,height=820,360
maxv=max(counts.max(),1)
bar_h=42
svg=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
     '<rect width="100%" height="100%" fill="white"/>',
     '<text x="24" y="32" font-family="Arial" font-size="20" font-weight="700">Action playbook mix</text>']
for i,(label,value) in enumerate(counts.items()):
    y=58+i*58
    bw=520*(value/maxv)
    svg.append(f'<text x="24" y="{y+27}" font-family="Arial" font-size="13">{label}</text>')
    svg.append(f'<rect x="250" y="{y+8}" width="{bw:.1f}" height="{bar_h-12}" rx="5" fill="#426B69"/>')
    svg.append(f'<text x="{260+bw:.1f}" y="{y+27}" font-family="Arial" font-size="13">{int(value):,}</text>')
svg.append('</svg>')
Path("work/figures/w07_action_mix.svg").write_text("\n".join(svg))

print(json.dumps(metrics,indent=2))
print("CSV: work/outputs/w07_action_queue.csv")
print("Figure: work/figures/w07_action_mix.svg")


{
  "rows_ranked": 30000,
  "top50_proxy_precision": 1.0,
  "top50_mean_score": 81.68250103193495,
  "action_counts": {
    "monitor_only": 12406,
    "review_title_meta_and_intent": 9759,
    "review_for_refresh": 4356,
    "review_engagement_and_intent": 3415,
    "review_depth_and_coverage": 64
  },
  "top_reason_counts": {
    "general_review": 12406,
    "low_ctr_visible": 9759,
    "model_high_risk": 8768,
    "low_engagement_visible": 6508,
    "thin_visible": 82,
    "stale_visible": 17
  },
  "monitor_policy": {
    "p50_floor": 0.6,
    "baseline_p50_reference": 0.26,
    "model_p50_reference": 0.86,
    "base_rate_reference": 0.3909677419354839,
    "base_rate_drift_absolute": 0.1,
    "review_if_model_loses_to_baseline_windows": 2
  },
  "human_review_required": true,
  "automatic_editing": false
}
CSV: work/outputs/w07_action_queue.csv
Figure: work/figures/w07_action_mix.svg


## Self-check

- [x] Ranked actions include human-readable reason codes
- [x] Intended use and limits are explicit
- [x] Human-review checklist and no-go automation list are explicit
- [x] Monitoring/retrain triggers are measurable
- [x] Queue CSV and paper-ready receipts/figure are exported
- [x] Claims remain decision-support, not causal
- [ ] Notebook executed top to bottom with visible outputs
- [ ] JSON receipt + figure committed
- [ ] Submit the public repository URL on the ML-10 card